<a href="https://colab.research.google.com/github/logonia/DAP/blob/main/pet_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# capfg_medical_final_v3.py - Fixed Dice computation with upsampling
# ============================================================

import os, random, time, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ---------------------------
# Check kagglehub (optional)
# ---------------------------
try:
    import kagglehub
    KAGGLEHUB_AVAILABLE = True
except ImportError:
    KAGGLEHUB_AVAILABLE = False

# ---------------------------
# Reproducibility & device
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------
# Helper functions
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

def dice_coefficient(pred, target, smooth=1e-6):
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    inter = (pred_flat * target_flat).sum()
    return (2. * inter + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

# ---------------------------
# Capsule components (same as before)
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.use_global = use_global
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        if use_global:
            self.global_pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        if self.use_global:
            g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
            logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy

class CapsuleNetAblation(nn.Module):
    def __init__(self, in_channels=3, shape=(32,32), classes=2, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.in_channels = in_channels
        self.spatial_shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(in_channels, 128, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        h_conv1 = shape[0] - 8
        w_conv1 = shape[1] - 8
        h_primary = (h_conv1 - 9) // 2 + 1
        w_primary = (w_conv1 - 9) // 2 + 1
        self.num_caps_in = 32 * h_primary * w_primary
        print(f"Computed input capsules: {self.num_caps_in} (spatial {h_primary}x{w_primary})")

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(128, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight, use_global=use_global)
        else:
            self.primary = PrimaryCapsuleBase(128, 32, 8)

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16 * classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, in_channels * shape[0] * shape[1]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))

        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)

        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)

        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]

        recon = self.decoder((out * y[:, :, None]).view(out.size(0), -1))
        recon = recon.view(-1, self.in_channels, self.spatial_shape[0], self.spatial_shape[1])

        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_entropy_loss(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# Dataset loaders (same as before, using 32x32)
# ---------------------------
class CheXmaskDataset(Dataset):
    def __init__(self, images_dir, masks_dir, target_size=(32,32)):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.target_size = target_size
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.png') or f.endswith('.jpg')])
        self.valid_pairs = []
        for img_f in self.image_files:
            base = os.path.splitext(img_f)[0]
            mask_path = os.path.join(masks_dir, base + '.png')
            if not os.path.exists(mask_path):
                mask_path = os.path.join(masks_dir, base + '_seg.png')
            if os.path.exists(mask_path):
                self.valid_pairs.append((os.path.join(images_dir, img_f), mask_path))
        print(f"Loaded {len(self.valid_pairs)} image-mask pairs.")
    def __len__(self):
        return len(self.valid_pairs)
    def __getitem__(self, idx):
        img_path, mask_path = self.valid_pairs[idx]
        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        img = img.resize(self.target_size, Image.BILINEAR)
        mask = mask.resize(self.target_size, Image.NEAREST)
        img = np.array(img).astype(np.float32) / 255.0
        mask = (np.array(mask) > 128).astype(np.float32)
        img = torch.from_numpy(img).permute(2,0,1)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return img, mask, 0, "chexmask"

def download_chexmask():
    if not KAGGLEHUB_AVAILABLE:
        raise ImportError("kagglehub not installed. Please run: pip install kagglehub")
    print("Downloading CheXmask dataset...")
    path = kagglehub.dataset_download("nstad/chexmask-cxr-segmentation-dataset")
    images_dir = os.path.join(path, 'images')
    masks_dir = os.path.join(path, 'masks')
    if not os.path.exists(images_dir):
        images_dir = os.path.join(path, 'png_images')
        masks_dir = os.path.join(path, 'png_masks')
    return images_dir, masks_dir

def load_chexmask(batch_size=8, val_split=0.2, target_size=(32,32)):
    images_dir, masks_dir = download_chexmask()
    dataset = CheXmaskDataset(images_dir, masks_dir, target_size)
    n = len(dataset)
    n_train = int(n * (1 - val_split))
    indices = list(range(n))
    random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    from torch.utils.data import Subset
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# Synthetic fallback
class SyntheticMedicalDataset(Dataset):
    def __init__(self, num_samples=2000, size=32):
        self.num_samples = num_samples
        self.size = size
        self.images = []
        self.masks = []
        for _ in range(num_samples):
            img = torch.randn(3, size, size) * 0.5
            mask = torch.zeros(1, size, size)
            cx, cy = random.randint(size//4, 3*size//4), random.randint(size//4, 3*size//4)
            r = random.randint(8, 20)
            y, x = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')
            circle = ((x - cx)**2 + (y - cy)**2) < r**2
            mask[:, circle] = 1.0
            img[:, circle] += 0.8
            self.images.append(img)
            self.masks.append(mask)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx], idx, 'synthetic'

def load_synthetic(batch_size=8, val_split=0.2, target_size=(32,32)):
    dataset = SyntheticMedicalDataset(num_samples=2000, size=target_size[0])
    n = len(dataset)
    n_train = int(n * (1 - val_split))
    indices = list(range(n))
    random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    from torch.utils.data import Subset
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# ---------------------------
# Training & evaluation (fixed Dice)
# ---------------------------
def evaluate_medical(model, loader, config, use_mask_loss=False):
    model.eval()
    total_loss, total_dice, total = 0.0, 0.0, 0
    with torch.no_grad():
        for x, mask, _, _ in loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, return_mask=True)   # pred_mask shape (B,1,Hm,Wm)
                # Upsample to mask size
                pred_mask_up = F.interpolate(pred_mask, size=mask.shape[-2:], mode='bilinear', align_corners=False)
                dice = dice_coefficient((pred_mask_up > 0.5).float(), mask)
                total_dice += dice.item() * x.size(0)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
                dice = 0.0
            total_loss += loss.item() * x.size(0)
            total += x.size(0)
    return total_loss/total, total_dice/total if total_dice>0 else 0.0

def train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss=False):
    optimizer = Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=config['lr_decay'])
    best_dice = 0.0
    patience, patience_counter = 5, 0
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        total = 0
        for x, mask, _, _ in train_loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            optimizer.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
            total += x.size(0)
        scheduler.step()
        val_loss, val_dice = evaluate_medical(model, val_loader, config, use_mask_loss)
        print(f"Epoch {epoch+1:02d}/{config['epochs']} | loss={epoch_loss/total:.4f} | val_dice={val_dice:.4f}")
        if val_dice > best_dice:
            best_dice = val_dice
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    return best_dice

def multi_run_medical(model_class, model_kwargs, config, train_loader, val_loader, n_runs=3, use_mask_loss=False):
    scores = []
    for run in range(n_runs):
        print(f"\n=== Run {run+1}/{n_runs} ===")
        set_seed(42+run)
        model = model_class(**model_kwargs).to(device)
        best = train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss)
        scores.append(best)
        print(f"Best Dice: {best:.4f}")
    return np.mean(scores), np.std(scores), scores

def plot_ablation(results, save_path='medical_ablation.png'):
    names = list(results.keys())
    means = [results[n][0] for n in names]
    stds = [results[n][1] for n in names]
    plt.figure(figsize=(8,5))
    plt.bar(names, means, yerr=stds, capsize=5, color=['gray','blue','green','red'])
    plt.ylabel('Dice Coefficient')
    plt.title('Medical Segmentation Ablation (resized to 32x32)')
    plt.ylim(0, 1.0)
    for i, (m, s) in enumerate(zip(means, stds)):
        plt.text(i, m+0.02, f'{m:.3f}±{s:.3f}', ha='center')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved plot to {save_path}")

# ---------------------------
# Main
# ---------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n_runs', type=int, default=2)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--target_size', type=int, nargs=2, default=[32,32])
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring unknown args: {unknown}")

    config = {
        'epochs': args.epochs,
        'batch_size': args.batch_size,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * (args.target_size[0] * args.target_size[1]),
        'classes': 2,
    }

    print("Loading medical dataset (resized to 32x32)...")
    try:
        train_loader, val_loader = load_chexmask(batch_size=args.batch_size, target_size=tuple(args.target_size))
    except Exception as e:
        print(f"CheXmask failed: {e}. Using synthetic fallback.")
        train_loader, val_loader = load_synthetic(batch_size=args.batch_size, target_size=tuple(args.target_size))

    experiments = [
        ('Baseline', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':False, 'use_capfg':False}, False),
        ('MaskOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':True, 'use_capfg':False, 'mask_threshold':0.1}, False),
        ('CapFGOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                       'use_input_mask':False, 'use_capfg':True,
                       'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True),
        ('Full', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                  'use_input_mask':True, 'use_capfg':True, 'mask_threshold':0.1,
                  'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True)
    ]

    print("\n" + "="*70)
    print(f"Ablation on Medical Data (n_runs={args.n_runs}, epochs={args.epochs})")
    print("="*70)

    results = {}
    for name, kwargs, use_loss in experiments:
        print(f"\n--- {name} ---")
        mean_dice, std_dice, _ = multi_run_medical(
            CapsuleNetAblation, kwargs, config,
            train_loader, val_loader, n_runs=args.n_runs, use_mask_loss=use_loss
        )
        results[name] = (mean_dice, std_dice)
        print(f"{name}: Dice = {mean_dice:.4f} ± {std_dice:.4f}")

    plot_ablation(results)
    df = pd.DataFrame([(n, results[n][0], results[n][1]) for n in results],
                      columns=['Model', 'Mean Dice', 'Std Dice'])
    df.to_csv('medical_ablation_results.csv', index=False)
    print("\nSaved medical_ablation_results.csv")
    print("✅ Done.")

if __name__ == '__main__':
    main()

Using device: cpu
Ignoring unknown args: ['-f', '/root/.local/share/jupyter/runtime/kernel-700d95cb-0e9e-4f75-9b90-e5911dce8aec.json']
Loading medical dataset (resized to 32x32)...
CheXmask failed: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.. Using synthetic fallback.

Ablation on Medical Data (n_runs=2, epochs=10)

--- Baseline ---

=== Run 1/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.5536 | val_dice=0.0000
Epoch 02/10 | loss=0.5633 | val_dice=0.0000
Epoch 03/10 | loss=0.5630 | val_dice=0.0000
Epoch 04/10 | loss=0.5628 | val_dice=0.0000
Epoch 05/10 | loss=0.5632 | val_dice=0.0000
Early stopping at epoch 5
Best Dice: 0.0000

=== Run 2/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.5648 | val_dice=0.0000
Epoch 02/10 | loss=0.56

In [ ]:
# ============================================================
# capfg_pets.py - Real-world segmentation on Oxford-IIIT Pets
# ============================================================

import os, random, time, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ---------------------------
# Reproducibility & device
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------
# Helper functions (capsule network)
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

def dice_coefficient(pred, target, smooth=1e-6):
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    inter = (pred_flat * target_flat).sum()
    return (2. * inter + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

# ---------------------------
# Capsule components (unchanged from your working code)
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.use_global = use_global
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        if use_global:
            self.global_pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        if self.use_global:
            g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
            logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy

class CapsuleNetAblation(nn.Module):
    def __init__(self, in_channels=3, shape=(32,32), classes=2, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.in_channels = in_channels
        self.spatial_shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(in_channels, 128, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        # compute number of primary capsules based on input size
        h_conv1 = shape[0] - 8
        w_conv1 = shape[1] - 8
        h_primary = (h_conv1 - 9) // 2 + 1
        w_primary = (w_conv1 - 9) // 2 + 1
        self.num_caps_in = 32 * h_primary * w_primary
        print(f"Computed input capsules: {self.num_caps_in} (spatial {h_primary}x{w_primary})")

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(128, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight, use_global=use_global)
        else:
            self.primary = PrimaryCapsuleBase(128, 32, 8)

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16 * classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, in_channels * shape[0] * shape[1]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))

        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)

        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)

        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]

        recon = self.decoder((out * y[:, :, None]).view(out.size(0), -1))
        recon = recon.view(-1, self.in_channels, self.spatial_shape[0], self.spatial_shape[1])

        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_entropy_loss(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# Oxford-IIIT Pet Dataset (real-world segmentation)
# ---------------------------
class PetDataset(Dataset):
    def __init__(self, root, split='train', target_size=(32,32)):
        self.target_size = target_size
        # Use torchvision's OxfordIIITPet to get images and masks
        self.dataset = datasets.OxfordIIITPet(root=root, split=split, download=True,
                                              target_types='segmentation',
                                              transform=transforms.ToTensor())
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, mask = self.dataset[idx]
        # img is already a tensor, shape (3, H, W) in [0,1]
        # mask is a PIL image where 0=background, 1=foreground, 2=border
        # We convert to binary mask: foreground = mask > 0
        mask = np.array(mask)
        mask = (mask > 0).astype(np.float32)
        # Resize image and mask to target size
        if img.shape[1:] != self.target_size:
            img = F.interpolate(img.unsqueeze(0), size=self.target_size, mode='bilinear', align_corners=False).squeeze(0)
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        if mask_t.shape[1:] != self.target_size:
            mask_t = F.interpolate(mask_t.unsqueeze(0), size=self.target_size, mode='nearest').squeeze(0)
        return img, mask_t, idx, 'pet'

def load_pet_dataset(batch_size=8, val_split=0.2, target_size=(32,32)):
    root = './data/oxford_pet'
    train_dataset = PetDataset(root, split='trainval', target_size=target_size)
    # Manually split because the dataset doesn't have a built-in validation split
    n = len(train_dataset)
    n_train = int(n * (1 - val_split))
    indices = list(range(n))
    random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    from torch.utils.data import Subset
    train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(train_dataset, val_idx), batch_size=batch_size, shuffle=False)
    print(f"Oxford Pets: {len(train_idx)} training, {len(val_idx)} validation")
    return train_loader, val_loader

# ---------------------------
# Training and evaluation
# ---------------------------
def evaluate_medical(model, loader, config, use_mask_loss=False):
    model.eval()
    total_loss, total_dice, total = 0.0, 0.0, 0
    with torch.no_grad():
        for x, mask, _, _ in loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, return_mask=True)
                pred_mask_up = F.interpolate(pred_mask, size=mask.shape[-2:], mode='bilinear', align_corners=False)
                dice = dice_coefficient((pred_mask_up > 0.5).float(), mask)
                total_dice += dice.item() * x.size(0)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
                dice = 0.0
            total_loss += loss.item() * x.size(0)
            total += x.size(0)
    return total_loss/total, total_dice/total if total_dice>0 else 0.0

def train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss=False):
    optimizer = Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=config['lr_decay'])
    best_dice = 0.0
    patience, patience_counter = 5, 0
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        total = 0
        for x, mask, _, _ in train_loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            optimizer.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
            total += x.size(0)
        scheduler.step()
        val_loss, val_dice = evaluate_medical(model, val_loader, config, use_mask_loss)
        print(f"Epoch {epoch+1:02d}/{config['epochs']} | loss={epoch_loss/total:.4f} | val_dice={val_dice:.4f}")
        if val_dice > best_dice:
            best_dice = val_dice
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    return best_dice

def multi_run_medical(model_class, model_kwargs, config, train_loader, val_loader, n_runs=3, use_mask_loss=False):
    scores = []
    for run in range(n_runs):
        print(f"\n=== Run {run+1}/{n_runs} ===")
        set_seed(42+run)
        model = model_class(**model_kwargs).to(device)
        best = train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss)
        scores.append(best)
        print(f"Best Dice: {best:.4f}")
    return np.mean(scores), np.std(scores), scores

def plot_ablation(results, save_path='pets_ablation.png'):
    names = list(results.keys())
    means = [results[n][0] for n in names]
    stds = [results[n][1] for n in names]
    plt.figure(figsize=(8,5))
    plt.bar(names, means, yerr=stds, capsize=5, color=['gray','blue','green','red'])
    plt.ylabel('Dice Coefficient')
    plt.title('Oxford-IIIT Pets Segmentation Ablation (32x32)')
    plt.ylim(0, 1.0)
    for i, (m, s) in enumerate(zip(means, stds)):
        plt.text(i, m+0.02, f'{m:.3f}±{s:.3f}', ha='center')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved plot to {save_path}")

# ---------------------------
# Main
# ---------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n_runs', type=int, default=2)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--target_size', type=int, nargs=2, default=[32,32])
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring unknown args: {unknown}")

    config = {
        'epochs': args.epochs,
        'batch_size': args.batch_size,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * (args.target_size[0] * args.target_size[1]),
        'classes': 2,
    }

    print("Loading Oxford-IIIT Pet dataset (real-world segmentation)...")
    train_loader, val_loader = load_pet_dataset(
        batch_size=args.batch_size,
        target_size=tuple(args.target_size)
    )

    experiments = [
        ('Baseline', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':False, 'use_capfg':False}, False),
        ('MaskOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':True, 'use_capfg':False, 'mask_threshold':0.1}, False),
        ('CapFGOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                       'use_input_mask':False, 'use_capfg':True,
                       'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True),
        ('Full', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                  'use_input_mask':True, 'use_capfg':True, 'mask_threshold':0.1,
                  'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True)
    ]

    print("\n" + "="*70)
    print(f"Ablation on Oxford-IIIT Pets (n_runs={args.n_runs}, epochs={args.epochs})")
    print("="*70)

    results = {}
    for name, kwargs, use_loss in experiments:
        print(f"\n--- {name} ---")
        mean_dice, std_dice, _ = multi_run_medical(
            CapsuleNetAblation, kwargs, config,
            train_loader, val_loader, n_runs=args.n_runs, use_mask_loss=use_loss
        )
        results[name] = (mean_dice, std_dice)
        print(f"{name}: Dice = {mean_dice:.4f} ± {std_dice:.4f}")

    plot_ablation(results)
    df = pd.DataFrame([(n, results[n][0], results[n][1]) for n in results],
                      columns=['Model', 'Mean Dice', 'Std Dice'])
    df.to_csv('pets_ablation_results.csv', index=False)
    print("\nSaved pets_ablation_results.csv")
    print("✅ Done (real-world pet segmentation).")

if __name__ == '__main__':
    main()

Using device: cpu
Ignoring unknown args: ['-f', '/root/.local/share/jupyter/runtime/kernel-700d95cb-0e9e-4f75-9b90-e5911dce8aec.json']
Loading Oxford-IIIT Pet dataset (real-world segmentation)...


100%|██████████| 792M/792M [00:17<00:00, 45.3MB/s]
100%|██████████| 19.2M/19.2M [00:00<00:00, 22.0MB/s]


Oxford Pets: 2944 training, 736 validation

Ablation on Oxford-IIIT Pets (n_runs=2, epochs=10)

--- Baseline ---

=== Run 1/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.0404 | val_dice=0.0000
Epoch 02/10 | loss=0.0309 | val_dice=0.0000
Epoch 03/10 | loss=0.0268 | val_dice=0.0000
Epoch 04/10 | loss=0.0220 | val_dice=0.0000
Epoch 05/10 | loss=0.0194 | val_dice=0.0000
Early stopping at epoch 5
Best Dice: 0.0000

=== Run 2/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.0405 | val_dice=0.0000
Epoch 02/10 | loss=0.0327 | val_dice=0.0000
Epoch 03/10 | loss=0.0256 | val_dice=0.0000
Epoch 04/10 | loss=0.0209 | val_dice=0.0000
Epoch 05/10 | loss=0.0188 | val_dice=0.0000
Early stopping at epoch 5
Best Dice: 0.0000
Baseline: Dice = 0.0000 ± 0.0000

--- MaskOnly ---

=== Run 1/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.0403 | val_dice=0.0000
Epoch 02/10 | loss=0.0311 | val_dice=0.0000
Epoch 03/10 | loss=0.0265 | val_dice=

In [2]:
# ============================================================
# capfg_pets.py - Real-world segmentation on Oxford-IIIT Pets
# ============================================================

import os, random, time, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ---------------------------
# Reproducibility & device
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------
# Helper functions (capsule network)
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

def dice_coefficient(pred, target, smooth=1e-6):
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    inter = (pred_flat * target_flat).sum()
    return (2. * inter + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

# ---------------------------
# Capsule components (unchanged from your working code)
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=128, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.use_global = use_global
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        if use_global:
            self.global_pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        if self.use_global:
            g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
            logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy

class CapsuleNetAblation(nn.Module):
    def __init__(self, in_channels=3, shape=(32,32), classes=2, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.in_channels = in_channels
        self.spatial_shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(in_channels, 128, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        # compute number of primary capsules based on input size
        h_conv1 = shape[0] - 8
        w_conv1 = shape[1] - 8
        h_primary = (h_conv1 - 9) // 2 + 1
        w_primary = (w_conv1 - 9) // 2 + 1
        self.num_caps_in = 32 * h_primary * w_primary
        print(f"Computed input capsules: {self.num_caps_in} (spatial {h_primary}x{w_primary})")

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(128, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight, use_global=use_global)
        else:
            self.primary = PrimaryCapsuleBase(128, 32, 8)

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16 * classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, in_channels * shape[0] * shape[1]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))

        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)

        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)

        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]

        recon = self.decoder((out * y[:, :, None]).view(out.size(0), -1))
        recon = recon.view(-1, self.in_channels, self.spatial_shape[0], self.spatial_shape[1])

        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_entropy_loss(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# Oxford-IIIT Pet Dataset (real-world segmentation)
# ---------------------------
class PetDataset(Dataset):
    def __init__(self, root, split='train', target_size=(32,32)):
        self.target_size = target_size
        # Use torchvision's OxfordIIITPet to get images and masks
        self.dataset = datasets.OxfordIIITPet(root=root, split=split, download=True,
                                              target_types='segmentation',
                                              transform=transforms.ToTensor())
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, mask = self.dataset[idx]
        # img is already a tensor, shape (3, H, W) in [0,1]
        # mask is a PIL image where 0=background, 1=foreground, 2=border
        # We convert to binary mask: foreground = mask > 0
        mask = np.array(mask)
        mask = (mask > 0).astype(np.float32)
        # Resize image and mask to target size
        if img.shape[1:] != self.target_size:
            img = F.interpolate(img.unsqueeze(0), size=self.target_size, mode='bilinear', align_corners=False).squeeze(0)
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        if mask_t.shape[1:] != self.target_size:
            mask_t = F.interpolate(mask_t.unsqueeze(0), size=self.target_size, mode='nearest').squeeze(0)
        return img, mask_t, idx, 'pet'

def load_pet_dataset(batch_size=8, val_split=0.2, target_size=(32,32)):
    root = './data/oxford_pet'
    train_dataset = PetDataset(root, split='trainval', target_size=target_size)
    # Manually split because the dataset doesn't have a built-in validation split
    n = len(train_dataset)
    n_train = int(n * (1 - val_split))
    indices = list(range(n))
    random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    from torch.utils.data import Subset
    train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(train_dataset, val_idx), batch_size=batch_size, shuffle=False)
    print(f"Oxford Pets: {len(train_idx)} training, {len(val_idx)} validation")
    return train_loader, val_loader

# ---------------------------
# Training and evaluation
# ---------------------------
def evaluate_medical(model, loader, config, use_mask_loss=False):
    model.eval()
    total_loss, total_dice, total = 0.0, 0.0, 0
    with torch.no_grad():
        for x, mask, _, _ in loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, return_mask=True)
                pred_mask_up = F.interpolate(pred_mask, size=mask.shape[-2:], mode='bilinear', align_corners=False)
                dice = dice_coefficient((pred_mask_up > 0.5).float(), mask)
                total_dice += dice.item() * x.size(0)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
                dice = 0.0
            total_loss += loss.item() * x.size(0)
            total += x.size(0)
    return total_loss/total, total_dice/total if total_dice>0 else 0.0

def train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss=False):
    optimizer = Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=config['lr_decay'])
    best_dice = 0.0
    patience, patience_counter = 5, 0
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        total = 0
        for x, mask, _, _ in train_loader:
            x, mask = x.to(device), mask.to(device)
            y_onehot = mask.view(mask.size(0), -1).mean(dim=1, keepdim=True).gt(0.5).float()
            optimizer.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, pred_mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(pred_mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
            total += x.size(0)
        scheduler.step()
        val_loss, val_dice = evaluate_medical(model, val_loader, config, use_mask_loss)
        print(f"Epoch {epoch+1:02d}/{config['epochs']} | loss={epoch_loss/total:.4f} | val_dice={val_dice:.4f}")
        if val_dice > best_dice:
            best_dice = val_dice
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    return best_dice

def multi_run_medical(model_class, model_kwargs, config, train_loader, val_loader, n_runs=3, use_mask_loss=False):
    scores = []
    for run in range(n_runs):
        print(f"\n=== Run {run+1}/{n_runs} ===")
        set_seed(42+run)
        model = model_class(**model_kwargs).to(device)
        best = train_medical_one_run(model, train_loader, val_loader, config, use_mask_loss)
        scores.append(best)
        print(f"Best Dice: {best:.4f}")
    return np.mean(scores), np.std(scores), scores

def plot_ablation(results, save_path='pets_ablation.png'):
    names = list(results.keys())
    means = [results[n][0] for n in names]
    stds = [results[n][1] for n in names]
    plt.figure(figsize=(8,5))
    plt.bar(names, means, yerr=stds, capsize=5, color=['gray','blue','green','red'])
    plt.ylabel('Dice Coefficient')
    plt.title('Oxford-IIIT Pets Segmentation Ablation (32x32)')
    plt.ylim(0, 1.0)
    for i, (m, s) in enumerate(zip(means, stds)):
        plt.text(i, m+0.02, f'{m:.3f}±{s:.3f}', ha='center')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved plot to {save_path}")

# ---------------------------
# Main
# ---------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n_runs', type=int, default=2)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--target_size', type=int, nargs=2, default=[32,32])
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring unknown args: {unknown}")

    config = {
        'epochs': args.epochs,
        'batch_size': args.batch_size,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * (args.target_size[0] * args.target_size[1]),
        'classes': 2,
    }

    print("Loading Oxford-IIIT Pet dataset (real-world segmentation)...")
    train_loader, val_loader = load_pet_dataset(
        batch_size=args.batch_size,
        target_size=tuple(args.target_size)
    )

    experiments = [
        ('Baseline', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':False, 'use_capfg':False}, False),
        ('MaskOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                      'use_input_mask':True, 'use_capfg':False, 'mask_threshold':0.1}, False),
        ('CapFGOnly', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                       'use_input_mask':False, 'use_capfg':True,
                       'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True),
        ('Full', {'in_channels':3, 'shape':tuple(args.target_size), 'classes':2,
                  'use_input_mask':True, 'use_capfg':True, 'mask_threshold':0.1,
                  'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, True)
    ]

    print("\n" + "="*70)
    print(f"Ablation on Oxford-IIIT Pets (n_runs={args.n_runs}, epochs={args.epochs})")
    print("="*70)

    results = {}
    for name, kwargs, use_loss in experiments:
        print(f"\n--- {name} ---")
        mean_dice, std_dice, _ = multi_run_medical(
            CapsuleNetAblation, kwargs, config,
            train_loader, val_loader, n_runs=args.n_runs, use_mask_loss=use_loss
        )
        results[name] = (mean_dice, std_dice)
        print(f"{name}: Dice = {mean_dice:.4f} ± {std_dice:.4f}")

    plot_ablation(results)
    df = pd.DataFrame([(n, results[n][0], results[n][1]) for n in results],
                      columns=['Model', 'Mean Dice', 'Std Dice'])
    df.to_csv('pets_ablation_results.csv', index=False)
    print("\nSaved pets_ablation_results.csv")
    print("✅ Done (real-world pet segmentation).")

if __name__ == '__main__':
    main()

Using device: cpu
Ignoring unknown args: ['-f', '/root/.local/share/jupyter/runtime/kernel-289d2e1c-f7c6-478d-9a97-756b51b6f6fb.json']
Loading Oxford-IIIT Pet dataset (real-world segmentation)...
Oxford Pets: 2944 training, 736 validation

Ablation on Oxford-IIIT Pets (n_runs=2, epochs=10)

--- Baseline ---

=== Run 1/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.0401 | val_dice=0.0000
Epoch 02/10 | loss=0.0295 | val_dice=0.0000
Epoch 03/10 | loss=0.0235 | val_dice=0.0000
Epoch 04/10 | loss=0.0200 | val_dice=0.0000
Epoch 05/10 | loss=0.0184 | val_dice=0.0000
Early stopping at epoch 5
Best Dice: 0.0000

=== Run 2/2 ===
Computed input capsules: 2048 (spatial 8x8)
Epoch 01/10 | loss=0.0403 | val_dice=0.0000
Epoch 02/10 | loss=0.0298 | val_dice=0.0000
Epoch 03/10 | loss=0.0228 | val_dice=0.0000
Epoch 04/10 | loss=0.0202 | val_dice=0.0000
Epoch 05/10 | loss=0.0183 | val_dice=0.0000
Early stopping at epoch 5
Best Dice: 0.0000
Baseline: Dice = 0.0000 ± 0.0000

--- Mas